# 🔧 Data Processing

**✍️ Author:** Hayriye Anıl  
**📘 Blog Series:** Time Series Analysis & Forecasting  

## 🎯 Purpose

This notebook merges weather datasets from separate years into a single dataset and performs essential data processing steps, including handling missing values and converting data types into their correct formats.

## 📂 Contents

- 📥 Data loading and initial inspection  
- ⏱️ Timestamp handling and resampling  
- 🧩 Missing data analysis and imputation  
- 📊 Example visualizations/code snippets for the first article (Understanding Time Series Data)

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

### Data loading and initial inspection

In [2]:
RAW_DATA_DIR = Path().resolve().parent / "data" / "raw_data"
PROCESSED_DATA_DIR = Path().resolve().parent / "data" / "processed_data"

In [3]:
files = list(RAW_DATA_DIR.glob('*.csv'))
files

[PosixPath('/Users/hayriyeanil/Documents/Documents/1_Projects/codespace/Time_Series_Blog_Series/data/raw_data/open-meteo-2022.csv'),
 PosixPath('/Users/hayriyeanil/Documents/Documents/1_Projects/codespace/Time_Series_Blog_Series/data/raw_data/open-meteo-2023.csv'),
 PosixPath('/Users/hayriyeanil/Documents/Documents/1_Projects/codespace/Time_Series_Blog_Series/data/raw_data/open-meteo-2024.csv'),
 PosixPath('/Users/hayriyeanil/Documents/Documents/1_Projects/codespace/Time_Series_Blog_Series/data/raw_data/open-meteo-2025.csv')]

In [4]:
data_frame = []
for file in files:
    data = pd.read_csv(file, sep=',', skiprows=3)
    data_frame.append(data)
    
dataset = pd.concat(data_frame, axis=0)
dataset

,time,temperature_2m (°C),relative_humidity_2m (%),dew_point_2m (°C),apparent_temperature (°C),rain (mm),weather_code (wmo code),wind_speed_10m (km/h),wind_speed_80m (km/h),wind_speed_120m (km/h),...,soil_temperature_0cm (°C),soil_temperature_6cm (°C),soil_temperature_18cm (°C),soil_temperature_54cm (°C),pressure_msl (hPa),surface_pressure (hPa),cloud_cover (%),cloud_cover_low (%),cloud_cover_mid (%),cloud_cover_high (%)
0,2022-01-01T00:00,8.1,85,5.7,6.7,0.0,3.0,2.0,2.5,2.7,...,NaN,NaN,NaN,NaN,1021.7,1017.7,100,5,42,100
1,2022-01-01T01:00,8.0,85,5.6,6.5,0.0,3.0,2.7,5.1,5.6,...,NaN,NaN,NaN,NaN,1021.4,1017.4,100,10,21,100
2,2022-01-01T02:00,7.8,85,5.5,6.3,0.0,2.0,2.5,5.1,5.2,...,NaN,NaN,NaN,NaN,1020.9,1016.9,68,5,58,55
3,2022-01-01T03:00,7.8,84,5.3,5.9,0.0,3.0,5.2,7.8,8.3,...,NaN,NaN,NaN,NaN,1021.1,1017.1,92,5,87,44
4,2022-01-01T04:00,7.8,85,5.4,5.6,0.0,3.0,7.2,13.7,14.5,...,NaN,NaN,NaN,NaN,1020.5,1016.5,100,5,52,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2025-12-31T19:00,1.9,67,-3.6,-1.6,0.0,0.0,5.5,8.4,11.2,...,4.7,3.4,6.6,9.8,1014.7,1010.7,0,0,0,0
8756,2025-12-31T20:00,1.7,66,-4.0,-1.6,0.0,0.0,4.4,6.8,9.2,...,4.6,2.8,6.5,9.8,1015.1,1011.1,0,0,0,0
8757,2025-12-31T21:00,1.4,66,-4.3,-1.9,0.0,0.0,3.8,5.1,6.9,...,4.2,2.1,6.3,9.8,1015.0,1011.0,0,0,0,0
8758,2025-12-31T22:00,1.5,62,-5.0,-1.9,0.0,0.0,4.1,4.9,6.3,...,4.0,1.8,6.1,9.8,1015.2,1011.2,0,0,0,0


In [5]:
dataset.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 35136 entries, 0 to 8759
Data columns (total 30 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   time                           35136 non-null  object 
 1   temperature_2m (°C)            35136 non-null  float64
 2   relative_humidity_2m (%)       35136 non-null  int64  
 3   dew_point_2m (°C)              35136 non-null  float64
 4   apparent_temperature (°C)      35136 non-null  float64
 5   rain (mm)                      35136 non-null  float64
 6   weather_code (wmo code)        35135 non-null  float64
 7   wind_speed_10m (km/h)          35136 non-null  float64
 8   wind_speed_80m (km/h)          35136 non-null  float64
 9   wind_speed_120m (km/h)         35136 non-null  float64
 10  wind_speed_180m (km/h)         27469 non-null  float64
 11  wind_direction_10m (°)         35136 non-null  int64  
 12  wind_direction_80m (°)         35136 non-null  int64

In [6]:
dataset.head(5)

,time,temperature_2m (°C),relative_humidity_2m (%),dew_point_2m (°C),apparent_temperature (°C),rain (mm),weather_code (wmo code),wind_speed_10m (km/h),wind_speed_80m (km/h),wind_speed_120m (km/h),...,soil_temperature_0cm (°C),soil_temperature_6cm (°C),soil_temperature_18cm (°C),soil_temperature_54cm (°C),pressure_msl (hPa),surface_pressure (hPa),cloud_cover (%),cloud_cover_low (%),cloud_cover_mid (%),cloud_cover_high (%)
0,2022-01-01T00:00,8.1,85,5.7,6.7,0.0,3.0,2.0,2.5,2.7,...,NaN,NaN,NaN,NaN,1021.7,1017.7,100,5,42,100
1,2022-01-01T01:00,8.0,85,5.6,6.5,0.0,3.0,2.7,5.1,5.6,...,NaN,NaN,NaN,NaN,1021.4,1017.4,100,10,21,100
2,2022-01-01T02:00,7.8,85,5.5,6.3,0.0,2.0,2.5,5.1,5.2,...,NaN,NaN,NaN,NaN,1020.9,1016.9,68,5,58,55
3,2022-01-01T03:00,7.8,84,5.3,5.9,0.0,3.0,5.2,7.8,8.3,...,NaN,NaN,NaN,NaN,1021.1,1017.1,92,5,87,44
4,2022-01-01T04:00,7.8,85,5.4,5.6,0.0,3.0,7.2,13.7,14.5,...,NaN,NaN,NaN,NaN,1020.5,1016.5,100,5,52,100


### Timestamp handling and resampling  

In [7]:
dataset['time'] = pd.to_datetime(dataset['time'], format='%Y-%m-%dT%H:%M')

In [8]:
dataset.set_index('time', inplace=True)

In [9]:
dataset

,temperature_2m (°C),relative_humidity_2m (%),dew_point_2m (°C),apparent_temperature (°C),rain (mm),weather_code (wmo code),wind_speed_10m (km/h),wind_speed_80m (km/h),wind_speed_120m (km/h),wind_speed_180m (km/h),...,soil_temperature_0cm (°C),soil_temperature_6cm (°C),soil_temperature_18cm (°C),soil_temperature_54cm (°C),pressure_msl (hPa),surface_pressure (hPa),cloud_cover (%),cloud_cover_low (%),cloud_cover_mid (%),cloud_cover_high (%)
time,,,,,,,,,,,,,,,,,,,,,
2022-01-01 00:00:00,8.1,85,5.7,6.7,0.0,3.0,2.0,2.5,2.7,NaN,...,NaN,NaN,NaN,NaN,1021.7,1017.7,100,5,42,100
2022-01-01 01:00:00,8.0,85,5.6,6.5,0.0,3.0,2.7,5.1,5.6,NaN,...,NaN,NaN,NaN,NaN,1021.4,1017.4,100,10,21,100
2022-01-01 02:00:00,7.8,85,5.5,6.3,0.0,2.0,2.5,5.1,5.2,NaN,...,NaN,NaN,NaN,NaN,1020.9,1016.9,68,5,58,55
2022-01-01 03:00:00,7.8,84,5.3,5.9,0.0,3.0,5.2,7.8,8.3,NaN,...,NaN,NaN,NaN,NaN,1021.1,1017.1,92,5,87,44
2022-01-01 04:00:00,7.8,85,5.4,5.6,0.0,3.0,7.2,13.7,14.5,NaN,...,NaN,NaN,NaN,NaN,1020.5,1016.5,100,5,52,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 19:00:00,1.9,67,-3.6,-1.6,0.0,0.0,5.5,8.4,11.2,15.5,...,4.7,3.4,6.6,9.8,1014.7,1010.7,0,0,0,0
2025-12-31 20:00:00,1.7,66,-4.0,-1.6,0.0,0.0,4.4,6.8,9.2,13.0,...,4.6,2.8,6.5,9.8,1015.1,1011.1,0,0,0,0
2025-12-31 21:00:00,1.4,66,-4.3,-1.9,0.0,0.0,3.8,5.1,6.9,9.6,...,4.2,2.1,6.3,9.8,1015.0,1011.0,0,0,0,0


### Missing data analysis and imputation  

In [10]:
dataset.isnull().sum()

temperature_2m (°C)                  0
relative_humidity_2m (%)             0
dew_point_2m (°C)                    0
apparent_temperature (°C)            0
rain (mm)                            0
weather_code (wmo code)              1
wind_speed_10m (km/h)                0
wind_speed_80m (km/h)                0
wind_speed_120m (km/h)               0
wind_speed_180m (km/h)            7667
wind_direction_10m (°)               0
wind_direction_80m (°)               0
wind_direction_120m (°)              0
wind_direction_180m (°)           7667
precipitation_probability (%)    30534
precipitation (mm)                   0
visibility (m)                       0
snow_depth (m)                       0
snowfall (cm)                        0
soil_temperature_0cm (°C)         7667
soil_temperature_6cm (°C)         7667
soil_temperature_18cm (°C)        7667
soil_temperature_54cm (°C)        7667
pressure_msl (hPa)                   0
surface_pressure (hPa)               0
cloud_cover (%)          

In [11]:
# Percantage of missing values in the dataset
dataset.isnull().mean() * 100

temperature_2m (°C)               0.000000
relative_humidity_2m (%)          0.000000
dew_point_2m (°C)                 0.000000
apparent_temperature (°C)         0.000000
rain (mm)                         0.000000
weather_code (wmo code)           0.002846
wind_speed_10m (km/h)             0.000000
wind_speed_80m (km/h)             0.000000
wind_speed_120m (km/h)            0.000000
wind_speed_180m (km/h)           21.820924
wind_direction_10m (°)            0.000000
wind_direction_80m (°)            0.000000
wind_direction_120m (°)           0.000000
wind_direction_180m (°)          21.820924
precipitation_probability (%)    86.902322
precipitation (mm)                0.000000
visibility (m)                    0.000000
snow_depth (m)                    0.000000
snowfall (cm)                     0.000000
soil_temperature_0cm (°C)        21.820924
soil_temperature_6cm (°C)        21.820924
soil_temperature_18cm (°C)       21.820924
soil_temperature_54cm (°C)       21.820924
pressure_ms

In [12]:
dataset[dataset["wind_speed_180m (km/h)"].isnull()][["wind_speed_180m (km/h)", "wind_direction_180m (°)", "soil_temperature_0cm (°C)", "soil_temperature_6cm (°C)", "soil_temperature_18cm (°C)", "soil_temperature_54cm (°C)" ]]

,wind_speed_180m (km/h),wind_direction_180m (°),soil_temperature_0cm (°C),soil_temperature_6cm (°C),soil_temperature_18cm (°C),soil_temperature_54cm (°C)
time,,,,,,
2022-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2022-01-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2022-11-16 06:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2022-11-16 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2022-11-16 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Remove "precipitation_probability (%)" and "weather_code (wmo code)" columns
dataset.drop(["precipitation_probability (%)", "weather_code (wmo code)"], axis=1, inplace=True)

In [14]:
# Start dataset from December 2022
dataset = dataset[(dataset.index >= "2022-12-01") & (dataset.index < "2026-01-01")]
dataset

,temperature_2m (°C),relative_humidity_2m (%),dew_point_2m (°C),apparent_temperature (°C),rain (mm),wind_speed_10m (km/h),wind_speed_80m (km/h),wind_speed_120m (km/h),wind_speed_180m (km/h),wind_direction_10m (°),...,soil_temperature_0cm (°C),soil_temperature_6cm (°C),soil_temperature_18cm (°C),soil_temperature_54cm (°C),pressure_msl (hPa),surface_pressure (hPa),cloud_cover (%),cloud_cover_low (%),cloud_cover_mid (%),cloud_cover_high (%)
time,,,,,,,,,,,,,,,,,,,,,
2022-12-01 00:00:00,13.1,86,10.8,10.4,0.0,20.5,34.4,39.5,44.6,39,...,13.2,12.8,13.3,14.2,1018.0,1014.1,100,87,97,87
2022-12-01 01:00:00,13.1,85,10.6,10.4,0.0,19.6,33.1,37.2,41.9,41,...,13.2,12.8,13.3,14.2,1017.8,1013.9,100,92,98,100
2022-12-01 02:00:00,12.9,86,10.6,10.5,0.3,17.7,29.3,32.8,36.7,38,...,13.1,12.8,13.3,14.2,1017.8,1013.9,100,96,100,100
2022-12-01 03:00:00,12.6,86,10.3,9.6,0.0,21.1,34.2,38.9,45.1,30,...,12.8,12.7,13.3,14.2,1018.0,1014.1,100,69,100,100
2022-12-01 04:00:00,12.5,88,10.6,9.5,0.0,21.8,35.5,40.1,45.4,31,...,12.8,12.5,13.3,14.2,1017.6,1013.7,100,61,100,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 19:00:00,1.9,67,-3.6,-1.6,0.0,5.5,8.4,11.2,15.5,337,...,4.7,3.4,6.6,9.8,1014.7,1010.7,0,0,0,0
2025-12-31 20:00:00,1.7,66,-4.0,-1.6,0.0,4.4,6.8,9.2,13.0,325,...,4.6,2.8,6.5,9.8,1015.1,1011.1,0,0,0,0
2025-12-31 21:00:00,1.4,66,-4.3,-1.9,0.0,3.8,5.1,6.9,9.6,319,...,4.2,2.1,6.3,9.8,1015.0,1011.0,0,0,0,0


In [15]:
dataset.head(5)

,temperature_2m (°C),relative_humidity_2m (%),dew_point_2m (°C),apparent_temperature (°C),rain (mm),wind_speed_10m (km/h),wind_speed_80m (km/h),wind_speed_120m (km/h),wind_speed_180m (km/h),wind_direction_10m (°),...,soil_temperature_0cm (°C),soil_temperature_6cm (°C),soil_temperature_18cm (°C),soil_temperature_54cm (°C),pressure_msl (hPa),surface_pressure (hPa),cloud_cover (%),cloud_cover_low (%),cloud_cover_mid (%),cloud_cover_high (%)
time,,,,,,,,,,,,,,,,,,,,,
2022-12-01 00:00:00,13.1,86,10.8,10.4,0.0,20.5,34.4,39.5,44.6,39,...,13.2,12.8,13.3,14.2,1018.0,1014.1,100,87,97,87
2022-12-01 01:00:00,13.1,85,10.6,10.4,0.0,19.6,33.1,37.2,41.9,41,...,13.2,12.8,13.3,14.2,1017.8,1013.9,100,92,98,100
2022-12-01 02:00:00,12.9,86,10.6,10.5,0.3,17.7,29.3,32.8,36.7,38,...,13.1,12.8,13.3,14.2,1017.8,1013.9,100,96,100,100
2022-12-01 03:00:00,12.6,86,10.3,9.6,0.0,21.1,34.2,38.9,45.1,30,...,12.8,12.7,13.3,14.2,1018.0,1014.1,100,69,100,100
2022-12-01 04:00:00,12.5,88,10.6,9.5,0.0,21.8,35.5,40.1,45.4,31,...,12.8,12.5,13.3,14.2,1017.6,1013.7,100,61,100,100


In [16]:
dataset.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 27120 entries, 2022-12-01 00:00:00 to 2025-12-31 23:00:00
Data columns (total 27 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   temperature_2m (°C)         27120 non-null  float64
 1   relative_humidity_2m (%)    27120 non-null  int64  
 2   dew_point_2m (°C)           27120 non-null  float64
 3   apparent_temperature (°C)   27120 non-null  float64
 4   rain (mm)                   27120 non-null  float64
 5   wind_speed_10m (km/h)       27120 non-null  float64
 6   wind_speed_80m (km/h)       27120 non-null  float64
 7   wind_speed_120m (km/h)      27120 non-null  float64
 8   wind_speed_180m (km/h)      27120 non-null  float64
 9   wind_direction_10m (°)      27120 non-null  int64  
 10  wind_direction_80m (°)      27120 non-null  int64  
 11  wind_direction_120m (°)     27120 non-null  int64  
 12  wind_direction_180m (°)     27120 non-null  float64
 

In [17]:
dataset.isnull().sum()

temperature_2m (°C)           0
relative_humidity_2m (%)      0
dew_point_2m (°C)             0
apparent_temperature (°C)     0
rain (mm)                     0
wind_speed_10m (km/h)         0
wind_speed_80m (km/h)         0
wind_speed_120m (km/h)        0
wind_speed_180m (km/h)        0
wind_direction_10m (°)        0
wind_direction_80m (°)        0
wind_direction_120m (°)       0
wind_direction_180m (°)       0
precipitation (mm)            0
visibility (m)                0
snow_depth (m)                0
snowfall (cm)                 0
soil_temperature_0cm (°C)     0
soil_temperature_6cm (°C)     0
soil_temperature_18cm (°C)    0
soil_temperature_54cm (°C)    0
pressure_msl (hPa)            0
surface_pressure (hPa)        0
cloud_cover (%)               0
cloud_cover_low (%)           0
cloud_cover_mid (%)           0
cloud_cover_high (%)          0
dtype: int64

In [18]:
dataset.to_csv(f"{PROCESSED_DATA_DIR}/processed_dataset.csv", index=True, header=True)

### Example visualizations/code snippets for the first article (Understanding Time Series Data)

In [19]:
dataset = pd.read_csv(f"{PROCESSED_DATA_DIR}/processed_dataset.csv", index_col=0, parse_dates=True)
dataset

,temperature_2m (°C),relative_humidity_2m (%),dew_point_2m (°C),apparent_temperature (°C),rain (mm),wind_speed_10m (km/h),wind_speed_80m (km/h),wind_speed_120m (km/h),wind_speed_180m (km/h),wind_direction_10m (°),...,soil_temperature_0cm (°C),soil_temperature_6cm (°C),soil_temperature_18cm (°C),soil_temperature_54cm (°C),pressure_msl (hPa),surface_pressure (hPa),cloud_cover (%),cloud_cover_low (%),cloud_cover_mid (%),cloud_cover_high (%)
time,,,,,,,,,,,,,,,,,,,,,
2022-12-01 00:00:00,13.1,86,10.8,10.4,0.0,20.5,34.4,39.5,44.6,39,...,13.2,12.8,13.3,14.2,1018.0,1014.1,100,87,97,87
2022-12-01 01:00:00,13.1,85,10.6,10.4,0.0,19.6,33.1,37.2,41.9,41,...,13.2,12.8,13.3,14.2,1017.8,1013.9,100,92,98,100
2022-12-01 02:00:00,12.9,86,10.6,10.5,0.3,17.7,29.3,32.8,36.7,38,...,13.1,12.8,13.3,14.2,1017.8,1013.9,100,96,100,100
2022-12-01 03:00:00,12.6,86,10.3,9.6,0.0,21.1,34.2,38.9,45.1,30,...,12.8,12.7,13.3,14.2,1018.0,1014.1,100,69,100,100
2022-12-01 04:00:00,12.5,88,10.6,9.5,0.0,21.8,35.5,40.1,45.4,31,...,12.8,12.5,13.3,14.2,1017.6,1013.7,100,61,100,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 19:00:00,1.9,67,-3.6,-1.6,0.0,5.5,8.4,11.2,15.5,337,...,4.7,3.4,6.6,9.8,1014.7,1010.7,0,0,0,0
2025-12-31 20:00:00,1.7,66,-4.0,-1.6,0.0,4.4,6.8,9.2,13.0,325,...,4.6,2.8,6.5,9.8,1015.1,1011.1,0,0,0,0
2025-12-31 21:00:00,1.4,66,-4.3,-1.9,0.0,3.8,5.1,6.9,9.6,319,...,4.2,2.1,6.3,9.8,1015.0,1011.0,0,0,0,0


#### Time Series Data

In [20]:
fig = px.line(dataset, x=dataset.index, y='temperature_2m (°C)', title='Istanbul Temperature Over Time')
fig.update_layout(xaxis_title='Time', yaxis_title='Temperature (°C)')
fig.show()

#### Cross-Section Data

In [21]:
students = ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank']
exam_scores = [85, 92, 78, 88, 94, 79]

fig = go.Figure(data=[go.Bar(x=students, y=exam_scores)])

fig.update_layout(
    title="University Entrance Exam Scores in a Given Year",
    xaxis_title="Students",
    yaxis_title="Exam Scores",
)
fig.show()

#### Understand Timestamp: Time zone and Time Scale

In [22]:
import pytz
from datetime import datetime

local_time = datetime(2025, 3, 26, 12, 0, 0) 
local_tz = pytz.timezone('Europe/Istanbul')

localized_time = local_tz.localize(local_time)

utc_time = localized_time.astimezone(pytz.utc)

print("Local time:", localized_time)
print("UTC time:", utc_time)


Local time: 2025-03-26 12:00:00+03:00
UTC time: 2025-03-26 09:00:00+00:00


#### Resampling

In [23]:
daily_temp = dataset.resample('D').agg({
    'apparent_temperature (°C)': 'mean',
})
daily_temp

,apparent_temperature (°C)
time,
2022-12-01,11.916667
2022-12-02,9.245833
2022-12-03,8.316667
2022-12-04,8.179167
2022-12-05,9.558333
...,...
2025-12-27,1.004167
2025-12-28,0.887500
2025-12-29,0.370833


In [24]:
daily_temp.reset_index(inplace=True)
hourly_dates = pd.date_range(start=daily_temp['time'].min(), end=daily_temp['time'].max(), freq='h')
df_hourly = daily_temp.set_index('time').reindex(hourly_dates, method=None)
df_hourly['apparent_temperature_interpolated'] = df_hourly['apparent_temperature (°C)'].interpolate(method='linear')
df_hourly

,apparent_temperature (°C),apparent_temperature_interpolated
2022-12-01 00:00:00,11.916667,11.916667
2022-12-01 01:00:00,NaN,11.805382
2022-12-01 02:00:00,NaN,11.694097
2022-12-01 03:00:00,NaN,11.582812
2022-12-01 04:00:00,NaN,11.471528
...,...,...
2025-12-30 20:00:00,NaN,-0.108333
2025-12-30 21:00:00,NaN,-0.275000
2025-12-30 22:00:00,NaN,-0.441667
2025-12-30 23:00:00,NaN,-0.608333
